# Image Footprint & Geolocation Overlay

**Source**: `grdl/example/geolocation/geolocation_overlay.py`

## Learning Objectives

- Compute image footprint (bounding polygon in lat/lon)
- Sample pixel grid and overlay coordinates on imagery
- Validate geolocation accuracy via visual inspection
- Export coordinates for external mapping tools (Google Maps, QGIS)

## Theory: Footprint Computation

**Image footprint**: The geographic polygon enclosing the image extent.

**GRDL workflow**:
1. Sample corner + edge pixels → lat/lon via geolocation
2. Construct polygon from boundary points
3. Overlay on slippy map or export as GeoJSON

**Use cases**:
- Catalog indexing (spatial search)
- Overlap detection (multi-image mosaics)
- Mission planning (coverage verification)

**Geolocation validation**: Sample a grid of pixels across the image and overlay both pixel (row, col) and geographic (lat, lon) coordinates to verify geolocation accuracy.

## Data Setup

**Path Configuration**: Set path to a SAR image (SICD or BIOMASS).

For this example, use any SICD .nitf file or BIOMASS L1 directory.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass  # Not running in IPython

# Validate GRDL version
import grdl
from packaging.version import Version

REQUIRED_VERSION = '0.6.1'
current_version = Version(grdl.__version__)
required_version = Version(REQUIRED_VERSION)

if current_version < required_version:
    raise RuntimeError(
        f"GRDL {REQUIRED_VERSION}+ required. Found {grdl.__version__}. "
        f"Update with: pip install --upgrade grdl"
    )

print(f"✓ GRDL {grdl.__version__} (>= {REQUIRED_VERSION})")

In [ ]:
from pathlib import Path
import numpy as np
import napari

from grdl.IO import get_reader
from grdl.geolocation import create_geolocation
from grdl.geolocation.chip import ChipGeolocation

# Configuration (update paths to your data)
SICD_PATH = '/path/to/your/SICD.nitf'  # ← CHANGE THIS to your file
FORMAT = 'sicd'  # or 'biomass'

# Processing options
CHIP_SIZE = 2048  # Crop large images to center chip (None = full image)
GRID_SIZE = 5  # Grid points per axis (5x5 = 25 points)
MARGIN_PCT = 5.0  # Margin from image edges (%)

print(f"Configuration:")
print(f"  Format: {FORMAT}")
print(f"  Chip size: {CHIP_SIZE if CHIP_SIZE else 'full image'}")
print(f"  Grid: {GRID_SIZE}x{GRID_SIZE} = {GRID_SIZE**2} points")
print(f"  Margin: {MARGIN_PCT}%")

In [ ]:
# Load image and build geolocation
# Extract a center chip to reduce memory and processing time
with get_reader(FORMAT, SICD_PATH) as reader:
    metadata = reader.metadata
    rows, cols = reader.get_shape()
    
    print(f"Full image shape: {rows} x {cols}")
    
    # Crop to center chip
    r_start = max(0, rows // 2 - CHIP_SIZE // 2)
    c_start = max(0, cols // 2 - CHIP_SIZE // 2)
    r_end = min(rows, r_start + CHIP_SIZE)
    c_end = min(cols, c_start + CHIP_SIZE)
    
    chip_data = reader.read_chip(r_start, r_end, c_start, c_end)
    chip_offset = (r_start, c_start)
    
    print(f"Reading center chip: {chip_data.shape[0]} x {chip_data.shape[1]}")
    print(f"  Chip offset: row={r_start}, col={c_start}")
    
    # Build geolocation (auto-detect from reader)
    geo_full = create_geolocation(reader)

# Wrap geolocation with chip offset
chip_shape = chip_data.shape[:2]
geo = ChipGeolocation(geo_full, row_offset=chip_offset[0], col_offset=chip_offset[1], shape=chip_shape)

print(f"\nGeolocation type: {type(geo).__name__}")

In [ ]:
# Compute grid of sample points across the chip
img_rows, img_cols = chip_data.shape[:2]
mr = int(img_rows * MARGIN_PCT / 100.0)
mc = int(img_cols * MARGIN_PCT / 100.0)

sample_rows = np.linspace(mr, img_rows - 1 - mr, GRID_SIZE, dtype=int)
sample_cols = np.linspace(mc, img_cols - 1 - mc, GRID_SIZE, dtype=int)

points = []
for r in sample_rows:
    for c in sample_cols:
        # Convert chip coordinates to full-image coordinates
        if isinstance(geo, ChipGeolocation):
            full_r = float(r + chip_offset[0])
            full_c = float(c + chip_offset[1])
        else:
            full_r, full_c = float(r), float(c)
        
        try:
            lat, lon, h = geo_full.image_to_latlon(full_r, full_c)
            if np.isfinite(lat) and np.isfinite(lon):
                points.append({
                    'img_row': r, 'img_col': c,
                    'full_row': full_r, 'full_col': full_c,
                    'lat': lat, 'lon': lon, 'height': h,
                })
        except (ValueError, NotImplementedError):
            continue

print(f"\nComputed {len(points)} grid points")
print(f"\nSample points (first 5):")
print(f"{'#':>3}  {'Row':>7}  {'Col':>7}  {'Latitude':>12}  {'Longitude':>12}  {'Height (m)':>10}")
print("-" * 62)
for i, p in enumerate(points[:5]):
    print(f"{i+1:3d}  {p['full_row']:7.0f}  {p['full_col']:7.0f}  "
          f"{p['lat']:12.6f}  {p['lon']:12.6f}  {p['height']:10.1f}")
if len(points) > 5:
    print(f"... ({len(points) - 5} more points)")

In [ ]:
# Convert to magnitude (dB) for visualization
def to_db(arr):
    return 20.0 * np.log10(np.abs(arr) + 1e-10)

db = to_db(chip_data)
vmin = np.nanpercentile(db, 2)
vmax = np.nanpercentile(db, 98)

print(f"\nImage statistics (dB):")
print(f"  Min: {db.min():.1f} dB")
print(f"  Max: {db.max():.1f} dB")
print(f"  Mean: {db.mean():.1f} dB")
print(f"  Display range: [{vmin:.1f}, {vmax:.1f}] dB (2nd-98th percentile)")

In [ ]:
# Visualize: overlay grid points with pixel and geographic coordinates using napari

# Close any existing viewer to avoid conflicts
try:
    viewer.close()
except (NameError, AttributeError, RuntimeError):
    pass

viewer = napari.Viewer(title="Image Footprint & Geolocation Overlay")

# Add base imagery
viewer.add_image(
    db,
    name="SAR Image (dB)",
    colormap="gray",
    contrast_limits=(vmin, vmax),
)

# Add grid points as markers
point_coords = np.array([[p['img_row'], p['img_col']] for p in points])

viewer.add_points(
    point_coords,
    name="Grid Points",
    size=20,
    face_color="cyan",
)

# Simplify viewer UI - remove all unnecessary controls for minimal display-only interface
viewer.window._qt_viewer.dockLayerControls.setVisible(False)
viewer.window._qt_viewer.dockLayerList.setVisible(False)
viewer.window._qt_viewer.dockConsole.setVisible(False)
try:
    viewer.window._qt_viewer.activityDock.setVisible(False)
except AttributeError:
    pass

print(f"\nnapari viewer opened — simple display mode")
print(f"\n{'='*70}")
print("WHAT YOU SHOULD SEE:")
print(f"{'='*70}")
print("1. Grayscale SAR image (magnitude in dB)")
print(f"2. {len(points)} cyan points overlaid in a {GRID_SIZE}×{GRID_SIZE} grid pattern")
print("\nTHIS VALIDATES:")
print("✓ Geolocation is working (pixels map to lat/lon)")
print("✓ Grid is regular (points are evenly spaced)")
print("✓ No coordinate flips or discontinuities")
print(f"\n{'='*70}")
print("Grid sample (pixel → geographic):")
print(f"{'='*70}")
for i, p in enumerate(points[:3]):
    print(f"Point {i+1}: ({p['full_row']:.0f}, {p['full_col']:.0f}) → "
          f"({p['lat']:.6f}°, {p['lon']:.6f}°)")
print(f"... ({len(points)-3} more points)")
print(f"\n{'='*70}")
print("Close the viewer window to continue...")
print(f"{'='*70}\n")

# Block until viewer is closed
napari.run()

# Automatic cleanup after viewer closes
print("\n✓ Viewer closed — cleaning up memory...")
del db, viewer
import gc
gc.collect()
print("✓ Memory cleanup complete")

In [ ]:
# Compute image footprint (bounding polygon)
# Sample corner + edge pixels
footprint_pixels = []
# Top edge
footprint_pixels.extend([(0, c) for c in range(0, img_cols, img_cols // 20)])
# Right edge
footprint_pixels.extend([(r, img_cols - 1) for r in range(0, img_rows, img_rows // 20)])
# Bottom edge
footprint_pixels.extend([(img_rows - 1, c) for c in range(img_cols - 1, -1, -(img_cols // 20))])
# Left edge
footprint_pixels.extend([(r, 0) for r in range(img_rows - 1, -1, -(img_rows // 20))])

footprint_coords = []
for r, c in footprint_pixels:
    try:
        if isinstance(geo, ChipGeolocation):
            full_r = float(r + chip_offset[0])
            full_c = float(c + chip_offset[1])
        else:
            full_r, full_c = float(r), float(c)
        lat, lon, _ = geo_full.image_to_latlon(full_r, full_c)
        if np.isfinite(lat) and np.isfinite(lon):
            footprint_coords.append((lon, lat))
    except (ValueError, NotImplementedError):
        continue

print(f"\nImage footprint: {len(footprint_coords)} boundary points")
print(f"Bounding box:")
lons = [c[0] for c in footprint_coords]
lats = [c[1] for c in footprint_coords]
print(f"  Latitude: [{min(lats):.6f}, {max(lats):.6f}]")
print(f"  Longitude: [{min(lons):.6f}, {max(lons):.6f}]")

In [ ]:
# Final cleanup
import gc

del chip_data, geo, geo_full, footprint_coords, metadata, points
gc.collect()
print("✓ Final memory cleanup complete")

## Physical Interpretation

**What the visualization shows:**
- **Grayscale image**: SAR magnitude data in dB (darker = weaker return, brighter = stronger return)
- **Cyan points**: Regular grid sampling across the image (every pixel has a valid lat/lon coordinate)

**Geolocation validation checks:**
1. **Grid regularity**: Points appear evenly spaced → geolocation is continuous (no warping)
2. **No discontinuities**: No sudden jumps between adjacent points
3. **Coverage**: Grid spans full image extent → geolocation works across entire scene

**Footprint**: The bounding polygon (computed below) defines the image's geographic extent on Earth. This is used for:
- Spatial catalog queries ("find all images covering lat/lon X")
- Multi-image overlap detection (mosaicking, change detection)
- Mission planning (coverage verification)

**What each grid point represents:**
```
Pixel (row, col) in image → Geographic (lat, lon) on WGS84 ellipsoid
```

For example, pixel (5000, 3000) might map to (34.052°N, 118.243°W) in Los Angeles.

**Export options** (optional follow-up cells):

For example, pixel (5000, 3000) might map to (34.052°N, 118.243°W) in Los Angeles.

**Export options** (optional follow-up cells):

**GeoJSON export** (for QGIS, web maps):
```python
import json
footprint_geojson = {
    "type": "Feature",
    "geometry": {
        "type": "Polygon",
        "coordinates": [footprint_coords]
    },
    "properties": {"format": FORMAT, "rows": rows, "cols": cols}
}
with open('footprint.geojson', 'w') as f:
    json.dump(footprint_geojson, f, indent=2)
```

**Google Maps link** (image center):
```python
center_lat = (min(lats) + max(lats)) / 2
center_lon = (min(lons) + max(lons)) / 2
print(f"https://www.google.com/maps?q={center_lat},{center_lon}&t=k")
```

## Summary

**GRDL patterns demonstrated**:
- ✅ `create_geolocation(reader)` — auto-detect geolocation from metadata
- ✅ `ChipGeolocation(geo, row_offset, col_offset, shape)` — chip offset wrapper for accurate sub-image geolocation
- ✅ `geo.image_to_latlon(row, col)` — pixel → lat/lon/height conversion
- ✅ Footprint computation — boundary sampling for geographic bounding polygon

**Validation workflow**: 
1. Sample regular pixel grid across image
2. Convert all pixels to lat/lon via geolocation
3. Visual inspection for continuity and regularity
4. Compute footprint polygon from boundary pixels
5. Export coordinates for external GIS tools